# Scrollie — MuSeg Thigh Segmentation Viewer

Pick a stack and scroll through slices comparing:
- **Left**: Dixon FAT image
- **Right**: MuSeg-AI thigh segmentation overlay

In [1]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
from ipywidgets import IntSlider, Dropdown, VBox
import ipywidgets as widgets
from IPython.display import display

In [8]:
SEG_DIR   = "museg_thigh_segs_dixon"
DATA_ROOT = "myosegmenTUM"

LABEL_MAP = {
    1:  "Vastus_Lateralis",
    2:  "Vastus_Intermedius",
    3:  "Vastus_Medialis",
    4:  "Rectus_Femoris",
    5:  "Sartorius",
    6:  "Gracilis",
    7:  "Semimembranosus",
    8:  "Semitendinosus",
    9:  "Biceps_Femoris",
    10: "Biceps_Femoris_Short",
    11: "Adductor_Magnus",
    12: "Adductor_Longus",
    13: "Adductor_Brevis",
}

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, "*_museg.nii.gz")))

def seg_to_nii(seg_path):
    # stem format: HV001_1_stack1
    stem    = os.path.basename(seg_path).replace("_museg.nii.gz", "")
    m       = re.match(r"(.+)_stack(\d+)$", stem)
    subject = m.group(1)
    stack_n = m.group(2)
    return os.path.join(DATA_ROOT, subject, "ImageData",
                        f"{subject}_FAT",
                        f"{subject}_FAT_stack{stack_n}.nii")

file_options = {
    os.path.basename(p).replace("_museg.nii.gz", ""): p
    for p in seg_files
}
print(f"Found {len(file_options)} segmented stacks")
if file_options:
    print(f"Image path example: {seg_to_nii(seg_files[0])}")

Found 46 segmented stacks
Image path example: myosegmenTUM\HV001_1\ImageData\HV001_1_FAT\HV001_1_FAT_stack1.nii


In [9]:
import matplotlib.cm as _cm
cmap = _cm.get_cmap("tab20", len(LABEL_MAP))

legend_patches = [
    mpatches.Patch(color=cmap(i), alpha=0.6, label=name)
    for i, (_, name) in enumerate(LABEL_MAP.items())
]

def build_overlay(seg_array):
    overlay = np.zeros((*seg_array.shape, 4), dtype=float)
    for i, (label_idx, _) in enumerate(LABEL_MAP.items()):
        color = cmap(i)
        overlay[seg_array == label_idx] = [color[0], color[1], color[2], 0.5]
    return overlay

def load_stack(label):
    seg_path = file_options[label]
    nii_path = seg_to_nii(seg_path)

    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)
    gt_norm   = (img_array - img_array.min()) / (img_array.max() - img_array.min() + 1e-8)

    seg_sitk  = sitk.ReadImage(seg_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk)
    overlay   = build_overlay(seg_array)

    return gt_norm, overlay

C:\Users\docto\AppData\Local\Temp\ipykernel_35824\2621501812.py:2: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = _cm.get_cmap("tab20", len(LABEL_MAP))


In [10]:
file_dropdown = Dropdown(options=list(file_options.keys()), description="Stack:")
slice_slider  = IntSlider(min=0, max=1, step=1, value=0, description="Slice:",
                          layout=widgets.Layout(width="600px"))
out = widgets.Output()

_cache = {}

def get_data(label):
    if label not in _cache:
        gt_norm, overlay = load_stack(label)
        _cache[label] = (gt_norm, overlay)
        slice_slider.max = gt_norm.shape[0] - 1
        slice_slider.value = 0
    return _cache[label]

def render(label, slice_idx):
    gt_norm, overlay = get_data(label)
    img = gt_norm[slice_idx]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(img, cmap="gray", origin="lower")
    axes[0].set_title(f"Image — slice {slice_idx}")
    axes[0].axis("off")

    axes[1].imshow(img, cmap="gray", origin="lower")
    axes[1].imshow(overlay[slice_idx], origin="lower")
    axes[1].set_title("MuSeg-AI Thigh Segmentation")
    axes[1].axis("off")
    axes[1].legend(handles=legend_patches, loc="lower right", fontsize=6, framealpha=0.7)

    fig.suptitle(label, fontsize=10)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()

def on_file_change(change):
    _cache.clear()
    get_data(change["new"])
    render(file_dropdown.value, slice_slider.value)

def on_slice_change(change):
    render(file_dropdown.value, change["new"])

file_dropdown.observe(on_file_change, names="value")
slice_slider.observe(on_slice_change, names="value")

if file_options:
    get_data(file_dropdown.value)
    render(file_dropdown.value, 0)

display(VBox([file_dropdown, slice_slider, out]))